# Adapting Generative Language Models

Five ways to get GPT-2 to perform the E2E NLG task (generating a restaurant description from a structured record), compared on the same data and the same prompt:

| Method | What is trained |
| --- | --- |
| Few-shot learning | nothing - only the examples in the prompt |
| LoRA | low-rank adapters in the attention layers |
| Prompt tuning | soft prompt vectors |
| Prefix-tuning | prefixes in every layer |
| P-tuning | a prompt produced by an LSTM encoder |

In [ ]:
!pip install transformers peft datasets
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from datasets import load_dataset
import json

data_files = {
    "train": "https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/trainset.csv",
    "validation": "https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/devset.csv"
}
dataset = load_dataset("csv", data_files=data_files)

train_size = 2000
val_size = 100

dataset_train = dataset['train'].shuffle(seed=42).select(range(train_size))
dataset_val = dataset['validation'].shuffle(seed=42).select(range(val_size))

print(dataset)
print("Text:", dataset_train[0]['ref'])
print("Attributes:", dataset_train[0]['mr'])

DatasetDict({
    train: Dataset({
        features: ['mr', 'ref'],
        num_rows: 42061
    })
    validation: Dataset({
        features: ['mr', 'ref'],
        num_rows: 4672
    })
})
Text: Located near the city centre is Clowns a low rated pub that costs more than 30.
Attributes: name[Clowns], eatType[pub], priceRange[more than £30], customer rating[low], area[city centre]


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, GenerationConfig
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_ckpt = 'openai-community/gpt2'

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def build_prompt(review_text, example_dataset, num_shots=3):
  prompt = "You are an expert on data extraction. Your task is to analyze restaurant opinions and turn them into structural format, extracting only critical information.\n"

  for i in range(num_shots):
    ex_text = example_dataset[i]['ref']
    ex_mr = example_dataset[i]['mr']

    prompt += f'\n{20*"-"}\nExample {i+1}\n{20*"-"}\nOpinion:\n{ex_text}\n\nYour output:\n{ex_mr}\n'

  prompt += f'\n{20*"-"}\nGenerate the structure\n{20*"-"}\nOpinion:\n{review_text}\n\nYour output:\n'
  return prompt

sample_text = dataset_val[0]['ref']
prompt = build_prompt(sample_text, dataset_train, num_shots=0)
print(prompt)

You are an expert on data extraction. Your task is to analyze restaurant opinions and turn them into structural format, extracting only critical information.

--------------------
Generate the structure
--------------------
Opinion:
Cocum is a coffee shop that serves Chinese food that is priced in the higher than average range. It has a customer rating of one out of five and is child-friendly.

Your output:



In [ ]:
def call_llm(prompt, model, tokenizer, device):
    generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)

    gen_config = GenerationConfig.from_pretrained(model_ckpt)
    gen_config.max_new_tokens = 100
    gen_config.max_length = None
    gen_config.pad_token_id = tokenizer.eos_token_id
    gen_config.eos_token_id = tokenizer.eos_token_id
    gen_config.do_sample = False
    gen_config.repetition_penalty = 1.15

    result = generator(
        prompt,
        generation_config=gen_config,
        do_sample=False,
        return_full_text=False
    )
    return result[0]['generated_text']

def generate_reponse(model, tokenizer, device, k=0):
  dataset_index = torch.randint(k, len(dataset_val), (1,)).item()
  sample_text = dataset_val[dataset_index]['ref']
  target_output = dataset_val[dataset_index]['mr']

  prompt = build_prompt(dataset_val[dataset_index]['ref'], dataset_train, num_shots=k)
  output = call_llm(prompt, model, tokenizer, device)

  print(f"Sample text:\n{sample_text}\n")
  print(50 * "=")
  print(f"Generated output:\n{output}\n")
  print(50 * "=")
  print(f"Target output:\n{target_output}\n")

## Few-shot learning

No training at all - 0, 3 and 8 examples in the prompt.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_ckpt)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
generate_reponse(model, tokenizer, device, 0)

Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Sample text:
Child-friendly venue The Wrestlers is rated 1 out of 5 by its customers.

Generated output:

1. The Wrestlers is a great venue for a live event.

2. The Wrestlers is a great venue for a live event.

3. The Wrestlers is a great venue for a live event.

4. The Wrestlers is a great venue for a live event.

5. The Wrestlers is a great venue for a live event.

6. The Wrestlers is a great venue for a live event.

7. The

Target output:
name[The Wrestlers], customer rating[1 out of 5], familyFriendly[yes]



In [ ]:
generate_reponse(model, tokenizer, device, 3)

Sample text:
The Cricketers is a coffee shop that is high rated, family friendly, and near The Portland Arms.

Generated output:

name[The Cricketers], eatType[coffee shop], food[Indian], priceRange[£20-25], customer rating[high], area[city centre], familyFriendly[no], near[Burger King]

--------------------

Generate the structure

--------------------

Opinion:

The Cricketers is a coffee shop that is high rated, family friendly, and near The Portland Arms.

Your output:

name[

Target output:
name[The Cricketers], eatType[coffee shop], food[English], customer rating[5 out of 5], familyFriendly[yes], near[The Portland Arms]



In [ ]:
generate_reponse(model, tokenizer, device, 8)

Sample text:
Located by the riverside, The Punter has a 5 out of 5 customer rating, and is not family-friendly.

Generated output:

name[The Punter], food[Japanese], priceRange[less than £20], familyFriendly[no]

--------------------
Example 9
--------------------

Opinion:
The Punter is a low priced Japanese restaurant by the riverside with a price range of £30-£40.

Your output:

name[The Punter], food[Japanese], priceRange[less than £30], familyFriendly[no]

--------------------
Example

Target output:
name[The Punter], customer rating[5 out of 5], area[riverside], familyFriendly[no]



## LoRA

Low-rank adapters injected into the attention layers. Only a fraction of the weights is trained; the rest of the model stays frozen.

In [ ]:
from transformers import DataCollatorForLanguageModeling

def format_and_tokenize(data_point):
  prompt = build_prompt(data_point["ref"], dataset_train, num_shots=0)
  prompt += data_point["mr"]
  prompt += tokenizer.eos_token

  tokenized_inputs = tokenizer(
      prompt,
      truncation=True,
      max_length=256
  )
  return tokenized_inputs

tokenized_train = dataset_train.map(format_and_tokenize, remove_columns=dataset_train.column_names)
tokenized_val = dataset_val.map(format_and_tokenize, remove_columns=dataset_val.column_names)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(tokenized_train[0])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

{'input_ids': [1639, 389, 281, 5887, 319, 1366, 22236, 13, 3406, 4876, 318, 284, 16602, 7072, 9317, 290, 1210, 606, 656, 13204, 5794, 11, 37895, 691, 4688, 1321, 13, 198, 198, 19351, 198, 8645, 378, 262, 4645, 198, 19351, 198, 46, 11635, 295, 25, 198, 43525, 1474, 262, 1748, 7372, 318, 36887, 82, 257, 1877, 13178, 2240, 326, 3484, 517, 621, 1542, 13, 198, 198, 7120, 5072, 25, 198, 3672, 58, 2601, 593, 82, 4357, 4483, 6030, 58, 12984, 4357, 2756, 17257, 58, 3549, 621, 4248, 1270, 4357, 6491, 7955, 58, 9319, 4357, 1989, 58, 19205, 7372, 60, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
print(tokenizer.decode(tokenized_train[1]["input_ids"]))
print("\n\n")
print(tokenizer.decode(tokenized_train[2]["input_ids"]))

You are an expert on data extraction. Your task is to analyze restaurant opinions and turn them into structural format, extracting only critical information.

--------------------
Generate the structure
--------------------
Opinion:
Near Burger King, at Riverside, there's a coffee shop that serves French food named The Eagle. It has a low customer rating, is not child-friendly, and it costs more than £30 to eat there.

Your output:
name[The Eagle], eatType[coffee shop], food[French], priceRange[more than £30], customer rating[low], area[riverside], familyFriendly[no], near[Burger King]<|endoftext|>



You are an expert on data extraction. Your task is to analyze restaurant opinions and turn them into structural format, extracting only critical information.

--------------------
Generate the structure
--------------------
Opinion:
The Eagle, a coffee shop that serves Indian food, has a price range of £20-£25. The Eagle has a high customer rating and it is situated in the city centre by 

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType


model = AutoModelForCausalLM.from_pretrained(model_ckpt)

lora_config = LoraConfig(
    r=8, lora_alpha=16, target_modules=["c_attn", "c_proj"], lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM
)
lora_model = get_peft_model(model, lora_config)


print("LoRA model parameters:")
lora_model.print_trainable_parameters()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


LoRA model parameters:
trainable params: 1,013,760 || all params: 125,453,568 || trainable%: 0.8081


In [ ]:
from transformers import Trainer, TrainingArguments
import math

training_args = TrainingArguments(
    output_dir="./gpt2-e2e-lora",          # Katalog na zapisywane wagi (checkpointy)
    per_device_train_batch_size=4,         # Rozmiar wsadu na GPU. Jeśli masz błąd OOM, zmniejsz do 2 lub 1.
    gradient_accumulation_steps=4,         # Akumulacja: wirtualnie zwiększa batch_size do 16 (4 * 4) przed wykonaniem optimizer.step()
    learning_rate=2e-4,                    # Dla algorytmów LoRA stosuje się standardowo wyższe wartości LR (np. 2e-4) niż przy pełnym fine-tuningu
    num_train_epochs=8,                    # Liczba pełnych przejść przez zbiór danych
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,                      # Częstotliwość wypisywania wartości Loss w konsoli
    fp16=True,                             # Uruchamia Automatic Mixed Precision (AMP) - trenowanie w 16 bitach
    optim="adamw_torch"                    # Standardowy optymalizator Adam z Weight Decay
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

steps_per_epoch = math.ceil(len(tokenized_train) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps))
total_steps = steps_per_epoch * training_args.num_train_epochs

print(f"Total optimization steps: {total_steps}")
print("Training...")
trainer.train()

Total optimization steps: 1000
Training...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
50,2.402635,1.923721
100,1.152907,0.964445
150,0.949932,0.838008
200,0.865825,0.768369
250,0.820158,0.728049
300,0.757099,0.695161
350,0.736707,0.672822
400,0.732175,0.658687
450,0.689175,0.639165
500,0.695349,0.625663


TrainOutput(global_step=1000, training_loss=0.8935688152313233, metrics={'train_runtime': 323.4839, 'train_samples_per_second': 49.462, 'train_steps_per_second': 3.091, 'total_flos': 995178701574144.0, 'train_loss': 0.8935688152313233, 'epoch': 8.0})

In [ ]:
generate_reponse(lora_model, tokenizer, device, 0)

Sample text:
There is a cheap coffee shop The Mill located in The Sorrento that provides breakfast.

Generated output:
name[The Mill], eatType[coffee shop], food[Fast food], priceRange[cheap], customer rating[low], area[riverside], familyFriendly[yes]

Your output:
name[The Mill], eatType[coffee shop], priceRange[cheap], customer rating[low], area[riverside], familyFriendly[yes]

Your output:
name[The Mill], food[Fast food], priceRange[

Target output:
name[The Mill], eatType[coffee shop], food[English], priceRange[less than £20], area[riverside], near[The Sorrento]



## Soft prompts

Instead of changing the model's weights, we learn continuous vectors prepended to the input.

### Prompt tuning

Vectors initialised from text, trained at the input only.

In [ ]:
from peft import PromptEmbedding, PromptTuningConfig, PromptTuningInit, get_peft_model

initial_prompt = "Hocus-pocus: 'name', 'eatType', 'familyFriendly', 'priceRange', 'food', 'near', 'area', 'customerRating'"
tokenized_initial_prompt = tokenizer(initial_prompt)
initial_prompt_len = len(tokenized_initial_prompt["input_ids"])

prompt_tuning_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text=initial_prompt,
    tokenizer_name_or_path=model_ckpt,
    num_virtual_tokens=initial_prompt_len
)
model = AutoModelForCausalLM.from_pretrained(model_ckpt)
prompt_tuning_model = get_peft_model(model, prompt_tuning_config)

print("Prompt-tuning model parameters:")
prompt_tuning_model.print_trainable_parameters()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prompt-tuning model parameters:
trainable params: 27,648 || all params: 124,467,456 || trainable%: 0.0222


In [ ]:
from transformers import Trainer, TrainingArguments
import math

training_args = TrainingArguments(
    output_dir="./gpt2-e2e-prompt-tuning",          # Katalog na zapisywane wagi (checkpointy)
    per_device_train_batch_size=4,         # Rozmiar wsadu na GPU. Jeśli masz błąd OOM, zmniejsz do 2 lub 1.
    gradient_accumulation_steps=4,         # Akumulacja: wirtualnie zwiększa batch_size do 16 (4 * 4) przed wykonaniem optimizer.step()
    learning_rate=0.1,                    # Wysoko, sygnał błędu (gradient) musi przejść przez całą zamrożoną sieć, zanim dotrze do wirtualnych tokenów, Prompt-tuning wymaga o rzędy wielkości wyższego współczynnika uczenia.
    num_train_epochs=8,                    # Liczba pełnych przejść przez zbiór danych
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,                      # Częstotliwość wypisywania wartości Loss w konsoli
    fp16=True,                             # Uruchamia Automatic Mixed Precision (AMP) - trenowanie w 16 bitach
    optim="adamw_torch"                    # Standardowy optymalizator Adam z Weight Decay
)

trainer = Trainer(
    model=prompt_tuning_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

steps_per_epoch = math.ceil(len(tokenized_train) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps))
total_steps = steps_per_epoch * training_args.num_train_epochs

print(f"Total optimization steps: {total_steps}")
print("Training...")
trainer.train()

Total optimization steps: 1000
Training...


Step,Training Loss,Validation Loss
50,3.213420,2.221620
100,2.111577,1.388655
150,1.698204,1.199017
200,1.510838,1.048331
250,1.409582,1.002758
300,1.286792,0.967755
350,1.229559,0.933814
400,1.238777,0.903509
450,1.181797,0.894144
500,1.168345,0.882149


TrainOutput(global_step=1000, training_loss=1.443491159439087, metrics={'train_runtime': 272.9726, 'train_samples_per_second': 58.614, 'train_steps_per_second': 3.663, 'total_flos': 985779307008000.0, 'train_loss': 1.443491159439087, 'epoch': 8.0})

In [ ]:
generate_reponse(prompt_tuning_model, tokenizer, device, 0)

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


Sample text:
There is a cheap coffee shop in the city centre named Fitzbillies. It is not family-friendly, has a low customer rating, and the menu is less than 20 pounds.

Generated output:
name[Fitzbilly], foodRange [cheap]Japanese], priceType['restaurant', averagePrice Range$20-$30], areaNearby) {city center}

       eatNumber[Cheaperbahce], nearThe Fence Cafe]
>This place serves Japanese cuisine with high prices. Its located at the riverside of Café Nairobi; it's very poor quality compared to French restaurants.[French]. your standard service:[no customers rate[

Target output:
name[Fitzbillies], eatType[coffee shop], food[English], priceRange[less than £20], customer rating[low], area[city centre], familyFriendly[no]



### Prefix-tuning

Prefixes added in every layer, not only at the input.

In [ ]:
from peft import PrefixTuningConfig, get_peft_model

prefix_tuning_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=32,
    prefix_projection=True
)
model = AutoModelForCausalLM.from_pretrained(model_ckpt)
prefix_tuning_model = get_peft_model(model, prefix_tuning_config)

print("Prefix-tuning model parameters:")
prefix_tuning_model.print_trainable_parameters()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prefix-tuning model parameters:
trainable params: 14,789,376 || all params: 139,229,184 || trainable%: 10.6223


In [ ]:
from transformers import Trainer, TrainingArguments
import math

training_args = TrainingArguments(
    output_dir="./gpt2-e2e-prefix-prompt-tuning",          # Katalog na zapisywane wagi (checkpointy)
    per_device_train_batch_size=4,         # Rozmiar wsadu na GPU. Jeśli masz błąd OOM, zmniejsz do 2 lub 1.
    gradient_accumulation_steps=4,         # Akumulacja: wirtualnie zwiększa batch_size do 16 (4 * 4) przed wykonaniem optimizer.step()
    learning_rate=1e-3,                    # Dla algorytmów LoRA stosuje się standardowo wyższe wartości LR (np. 2e-4) niż przy pełnym fine-tuningu
    num_train_epochs=8,                    # Liczba pełnych przejść przez zbiór danych
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,                      # Częstotliwość wypisywania wartości Loss w konsoli
    fp16=True,                             # Uruchamia Automatic Mixed Precision (AMP) - trenowanie w 16 bitach
    optim="adamw_torch"                    # Standardowy optymalizator Adam z Weight Decay
)

trainer = Trainer(
    model=prefix_tuning_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

steps_per_epoch = math.ceil(len(tokenized_train) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps))
total_steps = steps_per_epoch * training_args.num_train_epochs

print(f"Total optimization steps: {total_steps}")
print("Training...")
trainer.train()

Total optimization steps: 1000
Training...


Step,Training Loss,Validation Loss
50,2.314564,1.879850
100,2.287046,1.833263
150,2.221940,1.785797
200,2.144573,1.757510
250,2.145985,1.735433
300,2.044896,1.715933
350,2.042947,1.702151
400,2.089415,1.686973
450,1.990350,1.666603
500,2.019949,1.663655


TrainOutput(global_step=1000, training_loss=2.053338706970215, metrics={'train_runtime': 240.7912, 'train_samples_per_second': 66.448, 'train_steps_per_second': 4.153, 'total_flos': 985779307008000.0, 'train_loss': 2.053338706970215, 'epoch': 8.0})

In [ ]:
generate_reponse(prefix_tuning_model, tokenizer, device, 0)

Sample text:
Fitzbillies is a family friendly coffee shop in the city centre, offering cheap Chinese food and rated average by customers.

Generated output:
name[The Rice Boat], customer rating[yes] £20], priceRange[riverside], area[Cap], near[high], eatType[no], customer rating[coffee shop], customer rating[city centre], customer rating[5 out of 5], customer rating[The The located[The Golden], food[The Waterman], food[The has[The Strawberry[The Pearl], food[Thepub], food[The fast food[Theprice[The25],

Target output:
name[Fitzbillies], eatType[coffee shop], food[Chinese], priceRange[cheap], customer rating[average], area[city centre], familyFriendly[yes]



### P-tuning

The prompt is produced by a separate encoder (an LSTM) instead of learning raw vectors.

In [ ]:
from peft import PromptEncoderConfig, TaskType, get_peft_model, PromptEncoderReparameterizationType

p_tuning_config = PromptEncoderConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=64,
    encoder_hidden_size=256,
    encoder_reparameterization_type=PromptEncoderReparameterizationType.LSTM
)
model = AutoModelForCausalLM.from_pretrained(model_ckpt)
p_tuning_model = get_peft_model(model, p_tuning_config)

print("P-tuning model parameters:")
p_tuning_model.print_trainable_parameters()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


P-tuning model parameters:
trainable params: 4,384,000 || all params: 128,823,808 || trainable%: 3.4031


In [ ]:
from transformers import Trainer, TrainingArguments
import math

training_args = TrainingArguments(
    output_dir="./gpt2-e2e-p-tuning",          # Katalog na zapisywane wagi (checkpointy)
    per_device_train_batch_size=4,         # Rozmiar wsadu na GPU. Jeśli masz błąd OOM, zmniejsz do 2 lub 1.
    gradient_accumulation_steps=4,         # Akumulacja: wirtualnie zwiększa batch_size do 16 (4 * 4) przed wykonaniem optimizer.step()
    learning_rate=1e-2,                    # Dla algorytmów LoRA stosuje się standardowo wyższe wartości LR (np. 2e-4) niż przy pełnym fine-tuningu
    num_train_epochs=8,                    # Liczba pełnych przejść przez zbiór danych
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,                      # Częstotliwość wypisywania wartości Loss w konsoli
    fp16=True,                             # Uruchamia Automatic Mixed Precision (AMP) - trenowanie w 16 bitach
    optim="adamw_torch"                    # Standardowy optymalizator Adam z Weight Decay
)

trainer = Trainer(
    model=p_tuning_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

steps_per_epoch = math.ceil(len(tokenized_train) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps))
total_steps = steps_per_epoch * training_args.num_train_epochs

print(f"Total optimization steps: {total_steps}")
print("Training...")
trainer.train()

Total optimization steps: 1000
Training...


Step,Training Loss,Validation Loss
50,2.824398,2.231903
100,1.899581,1.462149
150,1.589861,1.215059
200,1.460582,1.127639
250,1.397136,1.078210
300,1.335528,1.056994
350,1.308773,1.031718
400,1.305081,1.026392
450,1.232164,0.975903
500,1.253152,0.961659


TrainOutput(global_step=1000, training_loss=1.43971697807312, metrics={'train_runtime': 372.4351, 'train_samples_per_second': 42.96, 'train_steps_per_second': 2.685, 'total_flos': 985779307008000.0, 'train_loss': 1.43971697807312, 'epoch': 8.0})

In [ ]:
generate_reponse(p_tuning_model, tokenizer, device, 0)

Sample text:
Fitzbillies is a family friendly coffee shop in the city centre, offering cheap Chinese food and rated average by customers.

Generated output:
name[Fitchburgers], eatType [coffee shops], priceRange[high]
categoryName[Chinese], customer rating,[average],"familyFriendly","price range", moderate,"is near"riverside"}




Target output:
name[Fitzbillies], eatType[coffee shop], food[Chinese], priceRange[cheap], customer rating[average], area[city centre], familyFriendly[yes]

